# 8.11 Attention mechanism — runnable notebook
Everything in the lesson, executed. Each concept has **3+ runnable examples** plus a static
visualization. Run-all safe and self-verifying (every cell ends in `assert`s). Only numpy,
matplotlib and torch — all preinstalled in Colab.

Map back to the lesson: **Scoring → QKᵀ → Scaling → Softmax → ×V → Multi-head → Masking → Full attention.**

In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.random.seed(0)
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)   # subtract max: numerical stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)
assert np.allclose(softmax(np.array([0., 0.])), [0.5, 0.5])
print("setup ok")

## Scoring — the dot product
The score is *just* a dot product: how aligned a Query is with a Key.

In [ ]:
a = np.array([2., 1.])
for name, k in [("aligned",[2,1]), ("orthogonal",[-1,2]), ("opposite",[-2,-1])]:
    print(f"{name:11} score = {a @ np.array(k):.0f}")
assert (a@np.array([2,1]), a@np.array([-1,2]), a@np.array([-2,-1])) == (5, 0, -5)

In [ ]:
k = np.array([3., 4.])
print("a.k =", a@k, "   (2a).k =", (2*a)@k)
assert (2*a)@k == 2*(a@k)   # magnitude scales the score

In [ ]:
def cos(u, v): return float(u@v / (np.linalg.norm(u)*np.linalg.norm(v)))
u = np.array([1., 0.])
for k in ([1,0],[10,0]):
    k = np.array(k, float); print(f"dot={u@k:5.1f}  cos={cos(u,k):.2f}")
assert cos(u, np.array([1.,0.])) == cos(u, np.array([10.,0.]))  # cosine ignores magnitude

In [ ]:
# viz: score = projection of Key (orange) onto Query (blue), swept over angle
fig, axes = plt.subplots(1, 5, figsize=(15, 3)); q = np.array([1., 0.])
for ax, ang in zip(axes, [0,45,90,135,180]):
    kk = np.array([np.cos(np.radians(ang)), np.sin(np.radians(ang))])
    ax.quiver(0,0,*q, angles='xy', scale_units='xy', scale=1, color='tab:blue')
    ax.quiver(0,0,*kk, angles='xy', scale_units='xy', scale=1, color='tab:orange')
    ax.set_xlim(-1.2,1.2); ax.set_ylim(-1.2,1.2); ax.set_aspect('equal')
    ax.set_title(f"{ang}°  q·k={q@kk:.2f}")
fig.suptitle("Score = alignment of Query (blue) and Key (orange)"); plt.tight_layout()

## The QKᵀ matrix and attention weights
Every Query dotted with every Key gives a T×T table; softmax each row into weights.

In [ ]:
Q = np.array([[1.,0.],[0.,1.],[1.,1.]]); K = np.array([[1.,0.],[1.,1.],[0.,1.]])
S = Q @ K.T
print(S)
assert S.shape == (3,3) and S[0,1] == 1 and S[1,0] == 0   # asymmetric: i->j != j->i

In [ ]:
A = softmax(S, axis=-1)
print(np.round(A, 3))
assert np.allclose(A.sum(1), 1)   # each row is a distribution

In [ ]:
def attn_flow(ax, A, left, right):
    T = A.shape[0]
    for i in range(T):
        for j in range(A.shape[1]):
            ax.plot([0,1],[T-1-i,T-1-j], lw=6*A[i,j], alpha=float(min(1,A[i,j])), color='tab:blue')
    for i,l in enumerate(left):  ax.text(-0.05,T-1-i,l,ha='right',va='center')
    for j,l in enumerate(right): ax.text(1.05,T-1-j,l,ha='left',va='center')
    ax.set_xlim(-0.6,1.6); ax.axis('off')
fig, ax = plt.subplots(figsize=(5,4))
attn_flow(ax, A, ["t0","t1","t2"], ["t0","t1","t2"])
ax.set_title("Attention flow (line opacity = weight)")

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
im = ax.imshow(A, cmap='Blues', vmin=0, vmax=1); plt.colorbar(im, ax=ax)
ax.set_title("QKᵀ softmax weights")

## Scaling by √d_k
Dot products grow like √d_k; without rescaling, softmax saturates and its gradient dies.

In [ ]:
rng = np.random.default_rng(0)
for dk in [4, 64, 512]:
    vals = [float(rng.standard_normal(dk) @ rng.standard_normal(dk)) for _ in range(5000)]
    print(f"dk={dk:4d}  std(q.k)={np.std(vals):6.2f}  ~ sqrt(dk)={dk**0.5:6.2f}")
    assert abs(np.std(vals) - dk**0.5) < 0.15 * dk**0.5

In [ ]:
s = np.array([2., 1., 0.]); big = s * 8   # as if unscaled at large dk
print("unscaled:", np.round(softmax(big), 3))
assert softmax(big).max() > 0.99   # near one-hot -> gradient ~ 0

In [ ]:
print("scaled  :", np.round(softmax(big/8), 3))
assert softmax(big/8).max() < 0.8   # soft, still learnable

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for dk, c in [(4,'tab:blue'),(64,'tab:orange'),(512,'tab:green')]:
    v = [float(rng.standard_normal(dk)@rng.standard_normal(dk)) for _ in range(3000)]
    axes[0].hist(v, bins=40, density=True, histtype='step', color=c, label=f"dk={dk}")
axes[0].legend(); axes[0].set_title("q·k spread grows with dk")
axes[1].bar(range(3), softmax(big), alpha=.5, label='unscaled')
axes[1].bar(range(3), softmax(big/8), alpha=.5, label='scaled')
axes[1].legend(); axes[1].set_title("unscaled spike vs scaled soft"); plt.tight_layout()

## Softmax
Nonnegative, sums to one, differentiable — and it *sharpens* score gaps.

In [ ]:
print(np.round(softmax(np.array([2.,1,0])), 3))
assert np.isclose(softmax(np.array([2.,1,0])).sum(), 1)

In [ ]:
assert np.allclose(softmax(np.array([2.,1,0])), softmax(np.array([102.,101,100])))
print("shift-invariant: [2,1,0] and [102,101,100] give the same weights")

In [ ]:
s = np.array([2., 1., 0.])
for T in [0.5, 1, 2]: print(f"T={T}: {np.round(softmax(s/T), 3)}")
assert softmax(s/0.5).max() > softmax(s/2).max()   # lower temperature -> peakier

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, T in zip(axes, [0.5, 1, 2]):
    ax.bar(range(3), softmax(s/T)); ax.set_ylim(0,1); ax.set_title(f"T={T}")
fig.suptitle("Softmax sharpens as temperature drops"); plt.tight_layout()

## The weighted average (×V)
The output is a weighted blend of Values — a convex combination, so it stays bounded.

In [ ]:
sc = np.array([4., 1., 2.5]); w = softmax(sc)   # the classic 'it' example
V = np.array([[0.,0.],[4.,0.],[2.,3.]])
out = w @ V
print("weights", np.round(w,3), "-> output", np.round(out,3))
assert np.allclose(np.round(w,3), [0.786, 0.039, 0.175])

In [ ]:
assert (w >= 0).all() and np.isclose(w.sum(), 1)   # convex combo => inside the hull
print("output is a convex combination of Values -> provably inside their hull")

In [ ]:
wt = np.array([0.01, 0.98, 0.01])
print(np.round(wt@V, 3), "~ V1 =", V[1])
assert np.allclose(wt@V, V[1], atol=0.1)   # near one-hot -> copies a single Value

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6)); tri = np.vstack([V, V[0]])
for ax, ww in zip(axes, [[1/3,1/3,1/3],[0.786,0.039,0.175],[0.01,0.98,0.01]]):
    ax.plot(tri[:,0], tri[:,1], 'k--', alpha=.5); ax.scatter(V[:,0], V[:,1], c='tab:blue')
    o = np.array(ww) @ V; ax.scatter(*o, c='tab:red', zorder=5)
    ax.set_title(f"w={np.round(ww,2)}"); ax.set_aspect('equal')
fig.suptitle("Output (red) always stays inside the Values' hull"); plt.tight_layout()

## Multi-head
Run several attentions in parallel subspaces; concatenate. Each head can track a different relation.

In [ ]:
d_model, h = 512, 8; dk = d_model // h
print("per-head dk =", dk, "  concat =", h*dk)
assert h * dk == d_model

In [ ]:
np.random.seed(3); X = np.random.randn(4, 8)
def head(X, seed):
    rng = np.random.default_rng(seed); d = X.shape[1]
    Wq, Wk, Wv = (rng.standard_normal((d,4)) for _ in range(3))
    Qh, Kh, Vh = X@Wq, X@Wk, X@Wv
    A = softmax(Qh @ Kh.T / 2, axis=-1)
    return A, A @ Vh
A1, _ = head(X, 1); A2, _ = head(X, 2)
print("head1 row0 argmax", A1[0].argmax(), " head2 row0 argmax", A2[0].argmax())
assert A1.shape == (4, 4) and np.allclose(A1.sum(1), 1)

In [ ]:
multi = np.concatenate([head(X, s)[1] for s in (1, 2)], axis=-1)
print("concatenated output shape:", multi.shape)
assert multi.shape == (4, 8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, (Ah, t) in zip(axes, [(A1,"head 1"), (A2,"head 2")]):
    ax.imshow(Ah, cmap='Blues', vmin=0, vmax=1); ax.set_title(t)
fig.suptitle("Two heads attend differently on the same input"); plt.tight_layout()

## Masking
Add −∞ *before* softmax so forbidden positions get exactly zero weight and rows still sum to one.

In [ ]:
np.random.seed(1); Sc = np.random.randn(4, 4)
mask = np.triu(np.full((4,4), -np.inf), 1)   # -inf strictly above the diagonal
Ac = softmax(Sc + mask, axis=-1)
print(np.round(Ac, 3))
assert np.allclose(np.triu(Ac, 1), 0) and np.allclose(Ac.sum(1), 1)

In [ ]:
row = np.array([1., 2, 0.5, 3]); pmask = np.array([0, 0, 0, -np.inf])   # last key is <pad>
a = softmax(row + pmask)
print(np.round(a, 3))
assert a[3] == 0 and np.isclose(a.sum(), 1)

In [ ]:
for i in range(4):
    assert np.allclose(Ac[i, i+1:], 0)   # position i never sees the future
print("each position attends only to indices <= itself")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, step in zip(axes, range(4)):
    M = np.zeros((4,4)); M[step] = softmax((Sc + mask)[step])
    ax.imshow(M, cmap='Blues', vmin=0, vmax=1); ax.set_title(f"generate token {step}")
fig.suptitle("Autoregressive: each step unlocks one more column"); plt.tight_layout()

## Full self-attention (capstone)
From scratch, cross-checked against PyTorch, then a selection demo on a toy sentence.

In [ ]:
def self_attention(X, Wq, Wk, Wv):
    Q, K, V = X@Wq, X@Wk, X@Wv
    A = softmax(Q @ K.T / np.sqrt(Q.shape[-1]), axis=-1)
    return A @ V, A
np.random.seed(0); T, d = 5, 8; X = np.random.randn(T, d)
Wq, Wk, Wv = (np.random.randn(d, d) for _ in range(3))
Wq, Wk, Wv = np.random.randn(d,d), np.random.randn(d,d), np.random.randn(d,d)
out_np, A_np = self_attention(X, Wq, Wk, Wv)
assert np.allclose(A_np.sum(1), 1); print("scratch output", out_np.shape)

In [ ]:
import torch, torch.nn.functional as F
Xt = torch.tensor(X)
Q, K, Vv = Xt @ torch.tensor(Wq), Xt @ torch.tensor(Wk), Xt @ torch.tensor(Wv)
A_t = F.softmax(Q @ K.T / (d ** 0.5), dim=-1)
assert np.allclose(A_t.numpy(), A_np, atol=1e-6)
print("PyTorch matches the from-scratch weights")

In [ ]:
tokens = ["the","animal","crossed","the","street","because","it","was","tired"]
n = len(tokens); Keys = np.eye(n)          # each token advertises a unique key
q_it = np.zeros(n); q_it[1] = 3.0          # 'it' queries strongly for 'animal' (index 1)
w_it = softmax(q_it @ Keys.T)
print("'it' attends most to:", tokens[int(w_it.argmax())])
assert tokens[int(w_it.argmax())] == "animal" 

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(A_np, cmap='Blues'); plt.colorbar(im, ax=ax)
ax.set_title("Full self-attention weights (5 tokens)")